# 🧬 Panacée — Entraînement Phase 3 (Kaggle)

**Phase 3** : Multi-propriétés (efficacité anti-VIH, solubilité, LogP, BBB, métabolisme)  
+ IA Raisonnement (combinaisons de molécules, synergie).

### Pré-requis :
- Avoir téléchargé `best_toxicity_model.pth` depuis la Phase 2
- Uploader ce fichier dans Kaggle : Ajouter en **Input** → *Upload a file*

### Avant de lancer :
1. Dashboard local lancé : `python -m webapp.run`
2. Ngrok actif : `ngrok http 8000`
3. Remplir `NGROK_URL`, `TOKEN`, `PHASE2_CKPT` ci-dessous

In [ ]:
# ╔══════════════════════════════════════════════╗
# ║  CONFIGURATION — modifier ces lignes ONLY    ║
# ╚══════════════════════════════════════════════╝

NGROK_URL   = "https://XXXX.ngrok-free.app"          # URL publique de votre dashboard
TOKEN       = "mon_token_secret_42"                   # == PANACEE_INGEST_TOKEN dans .env
RUN_NAME    = "kaggle_phase3_run01"                   # Nom affiché dans le dashboard
N_EPOCHS    = 30                                      # GPU: 30-80, CPU: 10-20

# Checkpoint Phase 2 uploadé dans Kaggle (chemin dans /kaggle/input/...)
PHASE2_CKPT = "/kaggle/input/panacee-phase2/best_toxicity_model.pth"

MAX_MOLECULES = None   # None = tous les datasets

print(f"Phase 2 checkpoint : {PHASE2_CKPT}")
print(f"Dashboard URL       : {NGROK_URL}")
print(f"Run name            : {RUN_NAME}")

In [ ]:
# ── Vérification connexion dashboard ────────────────────────────────────────
import urllib.request, json

try:
    r = urllib.request.urlopen(NGROK_URL.rstrip('/') + '/api/health', timeout=8)
    print(f"✅ Dashboard joignable — {json.loads(r.read())}")
except Exception as e:
    print(f"❌ Dashboard NON joignable : {e}")
    raise SystemExit("Corrigez la connexion avant de continuer.")

In [ ]:
import os, subprocess, sys
from pathlib import Path

# Vérifier le checkpoint Phase 2
if not Path(PHASE2_CKPT).exists():
    print(f"❌ Checkpoint Phase 2 introuvable : {PHASE2_CKPT}")
    print("   → Uploadez best_toxicity_model.pth en Input Kaggle")
    raise SystemExit("Checkpoint manquant.")
print(f"✅ Checkpoint Phase 2 OK ({Path(PHASE2_CKPT).stat().st_size / 1e6:.1f} MB)")

# Env vars
os.environ["PANACEE_PUSH_URL"]   = NGROK_URL
os.environ["PANACEE_PUSH_TOKEN"] = TOKEN
os.environ["PANACEE_PUSH_RUN"]   = RUN_NAME
os.environ["PYTHONIOENCODING"]   = "utf-8"
print("Variables d'environnement ✓")

In [ ]:
# ── Installation des dépendances ────────────────────────────────────────────
def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

pip("torch-geometric", "rdkit", "deepchem")
print("Dépendances OK ✓")

In [ ]:
# ── Cloner le repo ──────────────────────────────────────────────────────────
CLONE_DIR = Path("/kaggle/working/panacee")
REPO_URL  = "https://github.com/jumoras0000/savh_gnn.git"

if not CLONE_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(CLONE_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(CLONE_DIR), "pull", "--ff-only"], check=False)

PROJET = CLONE_DIR / "Projet_Panacee"
if str(PROJET) not in sys.path:
    sys.path.insert(0, str(PROJET))

# Copier le checkpoint Phase 2 là où Phase 3 l'attend
import shutil
SAVE_DIR = f"/kaggle/working/checkpoints/{RUN_NAME}"
Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)

print(f"Projet prêt : {PROJET}")

In [ ]:
# ── Lancer Phase 3 ──────────────────────────────────────────────────────────
import os as _os
_os.chdir(str(PROJET))

sys.argv = [
    "run_phase3.py",
    "--download",
    "--pretrained_model", PHASE2_CKPT,
    "--save_dir",         SAVE_DIR,
    "--epochs",           str(N_EPOCHS),
    "--skip_checks",      # pré-requis déjà vérifiés ci-dessus
]

print(f"Démarrage Phase 3 (run={RUN_NAME}, epochs={N_EPOCHS})...")
print(f"Push → {os.environ['PANACEE_PUSH_URL']}")
print()

from run_phase3 import main
main()

In [ ]:
# ── Résumé ──────────────────────────────────────────────────────────────────
import glob

pth_files = glob.glob(f"{SAVE_DIR}/**/*.pth", recursive=True)
print("Checkpoints sauvegardés :")
for f in sorted(pth_files):
    print(f"  {f}  ({Path(f).stat().st_size / 1e6:.1f} MB)")

print()
print("Télécharger depuis l'onglet Output Kaggle (à droite).")
print("Puis dans le dashboard → onglet Recherche → charger le checkpoint.")